# 05 - Auxiliary Sub-Task: Heat-Stress-Day Classification (SMOTE/ADASYN, scoped)

This is where SMOTE/ADASYN legitimately apply — see `reports/phase2_design.md` §5
for why they are *not* applied to the continuous DBT/WBT regression target. Scope
is deliberately kept secondary and small: a binary label (is a day a "high heat
stress" day, by WBT) is derived from the **training split only** (no leakage of
the threshold from val/test), then a small classifier is trained with and without
SMOTE/ADASYN rebalancing to see whether resampling actually helps at this scale.


In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (precision_score, recall_score, f1_score, roc_auc_score,
                              confusion_matrix, classification_report)
from imblearn.over_sampling import SMOTE, ADASYN
import matplotlib.pyplot as plt

pd.set_option("display.width", 120)

X_train = pd.read_csv("data/processed/X_train_flat.csv")
X_val = pd.read_csv("data/processed/X_val_flat.csv")
X_test = pd.read_csv("data/processed/X_test_flat.csv")
y_train = pd.read_csv("data/processed/y_train.csv")
y_val = pd.read_csv("data/processed/y_val.csv")
y_test = pd.read_csv("data/processed/y_test.csv")

## Label definition (threshold fit on train only)

`is_heat_stress_day = wbt > P80(wbt | train)`. The 80th percentile is a
deliberately simple, transparent choice standing in for a heat-stress-risk
threshold (a true wet-bulb heat-stress threshold is a physiological/climatological
question beyond this project's scope) — the point here is the *methodology*
(train-only threshold, then SMOTE/ADASYN scoped strictly to a genuine
classification target), not a claim of physiological calibration.


In [2]:
threshold = y_train["wbt"].quantile(0.80)
print(f"Train-derived WBT threshold (P80): {threshold:.2f} C")

def label(y):
    return (y["wbt"] > threshold).astype(int)

y_train_cls = label(y_train)
y_val_cls = label(y_val)
y_test_cls = label(y_test)

for name, y in [("train", y_train_cls), ("val", y_val_cls), ("test", y_test_cls)]:
    n_pos = y.sum()
    print(f"{name}: {n_pos}/{len(y)} positive ({n_pos/len(y):.1%}) - class imbalance ratio "
          f"{ (len(y)-n_pos) / max(n_pos,1):.1f}:1")

Train-derived WBT threshold (P80): 30.00 C
train: 40/210 positive (19.0%) - class imbalance ratio 4.2:1
val: 15/46 positive (32.6%) - class imbalance ratio 2.1:1
test: 11/43 positive (25.6%) - class imbalance ratio 2.9:1


## Scale, then resample (SMOTE/ADASYN touch TRAIN only)

Val/test are never resampled or synthesized — real, untouched, held out for final
evaluation, per the brief's requirement.


In [3]:
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

resamplers = {"none": None, "SMOTE": SMOTE(random_state=42, k_neighbors=5),
              "ADASYN": ADASYN(random_state=42, n_neighbors=5)}

resampled = {}
for name, sampler in resamplers.items():
    if sampler is None:
        resampled[name] = (X_train_s, y_train_cls.values)
    else:
        try:
            Xr, yr = sampler.fit_resample(X_train_s, y_train_cls.values)
            resampled[name] = (Xr, yr)
        except ValueError as e:
            print(f"{name} failed ({e}) - likely too few minority samples for k_neighbors; skipping")
    if name in resampled:
        Xr, yr = resampled[name]
        print(f"{name}: {len(yr)} rows, {yr.sum()} positive ({yr.mean():.1%})")

none: 210 rows, 40 positive (19.0%)
SMOTE: 340 rows, 170 positive (50.0%)
ADASYN: 334 rows, 164 positive (49.1%)


## Train a classifier with each resampling strategy, evaluate on the untouched test split


In [4]:
results = {}
for name, (Xr, yr) in resampled.items():
    clf = LogisticRegression(max_iter=1000, random_state=42)
    clf.fit(Xr, yr)
    pred = clf.predict(X_test_s)
    proba = clf.predict_proba(X_test_s)[:, 1]
    results[name] = {
        "precision": precision_score(y_test_cls, pred, zero_division=0),
        "recall": recall_score(y_test_cls, pred, zero_division=0),
        "f1": f1_score(y_test_cls, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test_cls, proba) if y_test_cls.nunique() > 1 else float("nan"),
    }

results_df = pd.DataFrame(results).T
results_df

        precision    recall        f1   roc_auc
none     0.368421  0.636364  0.466667  0.755682
SMOTE    0.384615  0.909091  0.540541  0.710227
ADASYN   0.400000  0.909091  0.555556  0.752841

## Discussion

Compares recall specifically — the class-imbalance-technique motivation is
catching the rare positive (heat-stress) days, not overall accuracy, which a
majority-class classifier can win trivially at this imbalance ratio.


In [5]:
best_recall = results_df["recall"].idxmax()
print(f"Best recall on rare/positive class: {best_recall} "
      f"(recall={results_df.loc[best_recall, 'recall']:.2f})")
print()
print("Note: with only", len(y_test_cls), "test rows and", int(y_test_cls.sum()),
      "positive test cases, these numbers carry wide uncertainty - treat the")
print("comparison as directional (does resampling help at all here?), not as a")
print("precise ranking.")

Best recall on rare/positive class: SMOTE (recall=0.91)

Note: with only 43 test rows and 11 positive test cases, these numbers carry wide uncertainty - treat the
comparison as directional (does resampling help at all here?), not as a
precise ranking.


In [6]:
import pickle
with open("data/processed/classification_results.pkl", "wb") as f:
    pickle.dump({"results_df": results_df, "threshold": threshold}, f)
print("Saved classification_results.pkl")

Saved classification_results.pkl
